# RandBox – IP102 – Continual Learning (Finetune 1 epoch)

## Thiết lập continual learning
| Task | Classes (0-indexed) | Số lớp |
|------|---------------------|--------|
| Task 1 | 0 – 26 | 27 |
| Task 2 | 27 – 51 | 25 |
| Task 3 | 52 – 76 | 25 |
| Task 4 | 77 – 101 | 25 |

## Checkpoint pretrain
| Task | Link |
|------|------|
| Task 1 | https://drive.google.com/file/d/1HjvHm7YQ9VMUbU5mDIGmXg8BWtDqmGGn/view |
| Task 2 | https://drive.google.com/file/d/1eAidoPpZh3Agm4hgBY9RP4zeZefnJmqJ/view |
| Task 3 | https://drive.google.com/file/d/1LW8_5DZbjURdWejWdMdT1mdK-5NU9Z4p/view |
| Task 4 | https://drive.google.com/file/d/1ljZA2DZCxPt5FDkqdpMUqTW04X23CSwb/view |

## Bước 0 – Cài đặt môi trường

> Phải bật **GPU T4 x2** trong Settings trước khi chạy.

In [1]:
import_line = "from randbox.open_world_coco_evaluation import OpenWorldCOCOEvaluator\n"

In [2]:
import os, sys, json, math, random, subprocess, shutil, importlib
from pathlib import Path
from collections import defaultdict

def run(cmd, cwd=None, check=True, extra_env=None):
    is_str = isinstance(cmd, str)
    print('>>>', cmd if is_str else ' '.join(str(c) for c in cmd))
    env = os.environ.copy()
    if extra_env:
        env.update(extra_env)
    return subprocess.run(cmd, shell=is_str, cwd=str(cwd) if cwd else None,
                          check=check, env=env)

# ── Verify GPU ─────────────────────────────────────────────────────────────────
import torch
print(f'Python     : {sys.version.split()[0]}')
print(f'PyTorch    : {torch.__version__}')
print(f'CUDA avail : {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU chưa được bật! Vào Settings → Accelerator → GPU T4 x2 rồi restart.')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')

# ── pip deps ───────────────────────────────────────────────────────────────────
run(f'{sys.executable} -m pip install -q -U pip setuptools wheel')
run(f'{sys.executable} -m pip install -q pycocotools opencv-python-headless fvcore iopath omegaconf ninja')

# ── detectron2 ─────────────────────────────────────────────────────────────────
D2_SRC = Path('/kaggle/working/detectron2_src')

def try_import_d2():
    if str(D2_SRC) not in sys.path:
        sys.path.insert(0, str(D2_SRC))
    importlib.invalidate_caches()
    import detectron2
    return detectron2

try:
    d2 = try_import_d2()
    print('detectron2:', d2.__version__)
except (ImportError, ModuleNotFoundError):
    print('\nCài detectron2 từ source...')
    build_env = {'CUDA_HOME': '/usr/local/cuda', 'FORCE_CUDA': '1',
                 'SETUPTOOLS_USE_DISTUTILS': 'local'}
    if not D2_SRC.exists():
        run('git clone --depth 1 https://github.com/facebookresearch/detectron2.git '
            '/kaggle/working/detectron2_src')
    installed = False
    try:
        run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-e', '.'],
            cwd=D2_SRC, extra_env=build_env)
        installed = True
    except subprocess.CalledProcessError:
        pass
    if not installed:
        run([sys.executable, 'setup.py', 'develop'], cwd=D2_SRC, extra_env=build_env)
    d2 = try_import_d2()
print('detectron2 OK:', d2.__version__)

# ── Clone RandBox ──────────────────────────────────────────────────────────────
REPO_DIR = Path('/kaggle/working/RandBox')
if not (REPO_DIR / 'train_net.py').exists():
    run('git clone --depth 1 https://github.com/nta2112/Ann-Randbox-Custom.git /kaggle/working/RandBox')
else:
    print('RandBox repo exists.')

# ── Patch 1: Disable EvalHook ──────────────────────────────────────────────────
# EvalHook luôn gọi eval 1 lần sau after_train() bất kể EVAL_PERIOD.
# Evaluator gốc là VOC-style (cần XML), không tương thích COCO JSON → KeyError.
# Fix: disable hoàn toàn. Eval riêng qua --eval-only nếu cần.
tn_path = REPO_DIR / 'train_net.py'
code = tn_path.read_text(encoding='utf-8')
OLD_HOOK = 'ret.append(hooks.EvalHook(cfg.TEST.EVAL_PERIOD, test_and_save_results))'
NEW_HOOK = '# [PATCHED] EvalHook disabled — incompatible with COCO JSON dataset\n        pass'
if OLD_HOOK in code:
    tn_path.write_text(code.replace(OLD_HOOK, NEW_HOOK), encoding='utf-8')
    print('Patch 1 OK: EvalHook disabled')
elif '[PATCHED]' in code:
    print('Patch 1: already applied')
else:
    print('[WARNING] EvalHook line not found — check train_net.py manually')


# Patch 2: Force real TEST.IMS_PER_BATCH support for eval-only



def patch_coco_json_evaluator(code):
    """Idempotent patch for train_net.py.
    - Ensures DatasetEvaluators + OpenWorldCOCOEvaluator are imported.
    - Does NOT rewrite build_evaluator; the repo version is already correct
      (uses OpenWorldCOCOEvaluator only, no COCOEvaluator that would crash
      with class=33 AssertionError on cross-task predictions).
    """
    # -- Import patch: add DatasetEvaluators + OpenWorldCOCOEvaluator (idempotent) --
    if 'DatasetEvaluators' not in code:
        code = code.replace(
            'from detectron2.evaluation import COCOEvaluator, LVISEvaluator, verify_results',
            'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results'
        )
    if 'open_world_coco_evaluation import OpenWorldCOCOEvaluator' not in code:
        lines = code.splitlines(keepends=True)
        new_lines = []
        for _l in lines:
            new_lines.append(_l)
            if _l.strip() == 'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results':
                new_lines.append('from randbox.open_world_coco_evaluation import OpenWorldCOCOEvaluator\n')
        code = ''.join(new_lines)
    return code
def ensure_eval_batch_patch():
    tn_path = REPO_DIR / 'train_net.py'
    code = tn_path.read_text(encoding='utf-8')
    # -- Import patch: add DatasetEvaluators + OpenWorldCOCOEvaluator (idempotent) --
    if 'DatasetEvaluators' not in code:
        code = code.replace(
            'from detectron2.evaluation import COCOEvaluator, LVISEvaluator, verify_results',
            'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results'
        )
    if 'open_world_coco_evaluation import OpenWorldCOCOEvaluator' not in code:
        lines = code.splitlines(keepends=True)
        new_lines = []
        for _l in lines:
            new_lines.append(_l)
            if _l.strip() == 'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results':
                new_lines.append('from randbox.open_world_coco_evaluation import OpenWorldCOCOEvaluator\n')
        code = ''.join(new_lines)

    code = code.replace(
        'from detectron2.data import build_detection_train_loader, build_detection_test_loader',
        'from detectron2.data import build_detection_train_loader'
    )
    code = code.replace(
        'from detectron2.data.common import DatasetFromList, MapDataset, trivial_batch_collator\n',
        'from detectron2.data.common import DatasetFromList, MapDataset\nfrom detectron2.data.build import trivial_batch_collator\n'
    )
    if 'get_detection_dataset_dicts' not in code:
        code = code.replace(
            'from detectron2.data import build_detection_train_loader',
            'from detectron2.data import build_detection_train_loader, get_detection_dataset_dicts'
        )
    if 'from detectron2.data.build import trivial_batch_collator' not in code:
        code = code.replace(
            'from detectron2.data.datasets.coco import load_coco_json\n',
            'from detectron2.data.datasets.coco import load_coco_json\n'
            'from detectron2.data.common import DatasetFromList, MapDataset\n'
            'from detectron2.data.build import trivial_batch_collator\n'
            'from detectron2.data.samplers import InferenceSampler\n'
        )
    if 'cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED]' not in code:
        code = code.replace(
            '    add_model_ema_configs(cfg)\n    cfg.merge_from_file(args.config_file)',
            '    add_model_ema_configs(cfg)\n    cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED] default eval batch size\n    cfg.merge_from_file(args.config_file)'
        )

    old_start = '    @classmethod\n    def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Use TEST.IMS_PER_BATCH for eval-only.\n'
    opt_start = '    @classmethod\n    def build_optimizer(cls, cfg, model):'
    if old_start in code:
        start = code.index(old_start)
        end = code.index(opt_start, start)
        code = code[:start] + code[end:]

    if 'def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Force real batched eval loader.' not in code:
        new_loader = (
            '    @classmethod\n'
            '    def build_test_loader(cls, cfg, dataset_name):\n'
            '        # [PATCHED] Force real batched eval loader.\n'
            '        dataset_dicts = get_detection_dataset_dicts(dataset_name, filter_empty=False)\n'
            '        mapper = RandBoxDatasetMapper(cfg, is_train=False)\n'
            '        dataset = MapDataset(DatasetFromList(dataset_dicts, copy=False), mapper)\n'
            '        world_size = max(1, comm.get_world_size())\n'
            '        total_batch = max(1, int(cfg.TEST.IMS_PER_BATCH))\n'
            '        per_gpu_batch = max(1, total_batch // world_size)\n'
            '        sampler = InferenceSampler(len(dataset))\n'
            '        batch_sampler = torch.utils.data.sampler.BatchSampler(\n'
            '            sampler, per_gpu_batch, drop_last=False\n'
            '        )\n'
            '        logger = logging.getLogger(__name__)\n'
            '        logger.info(\n'
            '            f"Eval loader: TEST.IMS_PER_BATCH={total_batch}, "\n'
            '            f"world_size={world_size}, per_gpu_batch={per_gpu_batch}, "\n'
            '            f"num_images={len(dataset)}, num_batches_per_rank={len(batch_sampler)}"\n'
            '        )\n'
            '        return torch.utils.data.DataLoader(\n'
            '            dataset,\n'
            '            num_workers=cfg.DATALOADER.NUM_WORKERS,\n'
            '            batch_sampler=batch_sampler,\n'
            '            collate_fn=trivial_batch_collator,\n'
            '        )\n\n'
            '    @classmethod\n'
            '    def build_optimizer(cls, cfg, model):'
        )
        code = code.replace('    @classmethod\n    def build_optimizer(cls, cfg, model):', new_loader)

    code = patch_coco_json_evaluator(code)
    tn_path.write_text(code, encoding='utf-8')
    patched = tn_path.read_text(encoding='utf-8')
    required = [
        'cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED]',
        'def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Force real batched eval loader.',
        'BatchSampler(',
    ]
    missing = [s for s in required if s not in patched]
    if missing:
        raise RuntimeError('Patch eval batch ch?a ?p d?ng ??: ' + repr(missing))
    print('Eval batch patch OK: real batched test loader is enabled')

ensure_eval_batch_patch()

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())
print('\nBước 0 hoàn tất.')

Python     : 3.12.13
PyTorch    : 2.10.0+cu128
CUDA avail : True
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB
>>> /usr/bin/python3 -m pip install -q -U pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.2 MB/s eta 0:00:00
>>> /usr/bin/python3 -m pip install -q pycocotools opencv-python-headless fvcore iopath omegaconf ninja

Cài detectron2 từ source...
>>> git clone --depth 1 https://github.com/facebookresearch/detectron2.git /kaggle/working/detectron2_src


Cloning into '/kaggle/working/detectron2_src'...


>>> /usr/bin/python3 -m pip install --no-build-isolation -e .
Obtaining file:///kaggle/working/detectron2_src
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for detectron2 (pyproject.toml): started
  Building editable for detectron2 (pyproject.toml): finished with status 'done'
  Created wheel for detectron2: filename=detectron2-0.6-0.editable-cp312-cp312-linux_x86_64.whl size=7323 sha256=4f915a68c1ec4aa2bd8ad564fd1247d84d53d4ce5df3e026e1bfc7a982f8377a
  Stored in directory: /tmp/pip-ephem-wheel-cache-8ienzawm/wheels/9c/9b/1f/011dee8c6d205a58cf1a663ede9a1ad5c484de18976d774ef4
Successfully built detectron2
  Attempting uninstall: iopath
    Found existing installation: iopath 0.1.10
    Uninstalling iopath-0.1.10:
      Successfully uninstalled

Cloning into '/kaggle/working/RandBox'...


Patch 1 OK: EvalHook disabled
Eval batch patch OK: real batched test loader is enabled
CWD: /kaggle/working/RandBox

Bước 0 hoàn tất.


## Bước 1 – Cấu hình

In [3]:
DATASET_ANN = Path('/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations')

# QUAN TRỌNG: trỏ đến 'classification/' (thư mục cha)
# file_name JSON = 'train/26/22165.jpg' → ghép thành classification/train/26/22165.jpg ✓
IMAGE_ROOT = Path('/kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification')

# Checkpoint pretrain của tác giả. Ví dụ:
# '/kaggle/input/randbox-pretrain/pytorch/default/1/model_0019999.pth'
PRETRAIN_WEIGHTS = "/kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.pth"   # ← ĐỔI THÀNH PATH CỦA BẠN

# ── Tốc độ ─────────────────────────────────────────────────────────────────────
NUM_GPUS      = torch.cuda.device_count()   # 2
IMS_PER_BATCH = 18   # 2×T4 + AMP: 12 tốt hơn 8 (~33% nhanh hơn)
NUM_WORKERS   = 4   # tăng từ 4 lên 8 để prefetch nhanh hơn

# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT_ROOT   = Path('/kaggle/working/randbox_outputs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DATASETS = REPO_DIR / 'datasets'

print(f'NUM_GPUS         = {NUM_GPUS}')
print(f'IMS_PER_BATCH    = {IMS_PER_BATCH}')
print(f'NUM_WORKERS      = {NUM_WORKERS}')
print(f'IMAGE_ROOT       = {IMAGE_ROOT}')
print(f'PRETRAIN_WEIGHTS = {PRETRAIN_WEIGHTS}')

NUM_GPUS         = 2
IMS_PER_BATCH    = 18
NUM_WORKERS      = 4
IMAGE_ROOT       = /kaggle/input/datasets/rtlmhjbn/ip02-dataset/classification
PRETRAIN_WEIGHTS = /kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.pth


## Bước 2 – Định nghĩa task

In [4]:
NUM_CLASSES_TOTAL = 103  # 102 known + 1 unknown slot

TASKS = {
    't1':    {'start': 0,  'end': 27,  'prev_cls': 0,  'curr_cls': 27, 'config': 'configs/t1.yaml'},
    't2_ft': {'start': 27, 'end': 52,  'prev_cls': 27, 'curr_cls': 25, 'config': 'configs/t2_ft.yaml'},
    't3_ft': {'start': 52, 'end': 77,  'prev_cls': 52, 'curr_cls': 25, 'config': 'configs/t3_ft.yaml'},
    't4_ft': {'start': 77, 'end': 102, 'prev_cls': 77, 'curr_cls': 25, 'config': 'configs/t4_ft.yaml'},
}

ACTIVE_TASKS = ['t1']
# ACTIVE_TASKS = ['t1', 't2_ft', 't3_ft', 't4_ft']  # 4 task liên tục

print('Active tasks:', ACTIVE_TASKS)

Active tasks: ['t1']


## Bước 3 – Tạo dataset split (50% mỗi lớp, đủ lớp)

In [5]:
random.seed(42)

SAMPLE_RATIO = 0.4   # cắt 60%, giữ lại 40% dữ liệu

def load_coco(path):
    with open(path) as f: return json.load(f)

def dump_coco(obj, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f: json.dump(obj, f)

def make_id_map(coco):
    cats = sorted(coco['categories'], key=lambda c: c['id'])
    return {c['id']: i for i, c in enumerate(cats)}, cats

def sample_images(annotations, ratio):
    """
    Giữ `ratio` % ảnh của mỗi class (tối thiểu 1 ảnh/class).
    Trả về set image_id đã chọn — đảm bảo đủ tất cả lớp.
    """
    if ratio >= 1.0:
        return {a['image_id'] for a in annotations}

    cat_imgs = defaultdict(list)
    for a in annotations:
        cat_imgs[a['category_id']].append(a['image_id'])

    selected = set()
    for cat_id, img_ids in cat_imgs.items():
        img_ids = sorted(set(img_ids))  # deduplicate + deterministic order
        n_keep = max(1, math.ceil(len(img_ids) * ratio))
        selected.update(random.sample(img_ids, n_keep))

    return selected

def build_train_split(split, task_start, task_end, unk, ratio=1.0):
    coco = load_coco(DATASET_ANN / f'{split}.json')
    g2i, all_cats = make_id_map(coco)
    task_cats = all_cats[task_start:task_end]
    sel_cat_ids = {c['id'] for c in task_cats}

    # Lọc annotations thuộc task này
    task_anns = [a for a in coco['annotations'] if a['category_id'] in sel_cat_ids]

    # Sample 50% ảnh mỗi lớp
    keep_imgs = sample_images(task_anns, ratio)

    new_cats = [{'id': g2i[c['id']], 'name': c['name'],
                 'supercategory': c.get('supercategory', '')} for c in task_cats]
    new_anns  = [dict(a, category_id=g2i[a['category_id']])
                 for a in task_anns if a['image_id'] in keep_imgs]
    new_imgs  = [i for i in coco['images'] if i['id'] in keep_imgs]

    return {'info': coco.get('info', {}), 'licenses': coco.get('licenses', []),
            'categories': new_cats, 'images': new_imgs, 'annotations': new_anns}

def build_test_split(split, task_start, task_end, unk, ratio=1.0):
    """Test: class ngoài known range → remap sang unknown slot."""
    coco = load_coco(DATASET_ANN / f'{split}.json')
    g2i, all_cats = make_id_map(coco)
    known = set(range(task_start, task_end))
    task_cats = all_cats[task_start:task_end]

    # Remap tất cả annotations (known giữ nguyên, ngoài range → unknown slot)
    remapped = []
    for a in coco['annotations']:
        g = g2i.get(a['category_id'])
        if g is None: continue
        remapped.append(dict(a, category_id=g if g in known else unk))

    # Sample 50% ảnh mỗi lớp (bao gồm cả unknown)
    keep_imgs = sample_images(remapped, ratio)

    new_cats = [{'id': g2i[c['id']], 'name': c['name'],
                 'supercategory': c.get('supercategory', '')} for c in task_cats]
    new_cats.append({'id': unk, 'name': 'unknown', 'supercategory': 'unknown'})
    new_anns = [a for a in remapped if a['image_id'] in keep_imgs]
    new_imgs = [i for i in coco['images'] if i['id'] in keep_imgs]

    return {'info': coco.get('info', {}), 'licenses': coco.get('licenses', []),
            'categories': new_cats, 'images': new_imgs, 'annotations': new_anns}


unk = NUM_CLASSES_TOTAL - 1  # = 102

for task_name in ACTIVE_TASKS:
    info = TASKS[task_name]
    ann_dir = WORK_DATASETS / task_name / 'annotations'
    img_dir = WORK_DATASETS / task_name / 'images'
    ann_dir.mkdir(parents=True, exist_ok=True)
    img_dir.mkdir(parents=True, exist_ok=True)

    # ── Train split ───────────────────────────────────────────────────────────
    tr = build_train_split('train', info['start'], info['end'], unk, SAMPLE_RATIO)
    dump_coco(tr, ann_dir / 'train.json')
    cls_in_train = len({a['category_id'] for a in tr['annotations']})
    print(f'{task_name}/train: {len(tr["images"]):5d} imgs  '
          f'{len(tr["annotations"]):6d} anns  '
          f'{cls_in_train} classes  (ratio={SAMPLE_RATIO})')

    # ── Test split ────────────────────────────────────────────────────────────
    te = build_test_split('test', info['start'], info['end'], unk, SAMPLE_RATIO)
    dump_coco(te, ann_dir / 'test.json')
    n_unk = sum(1 for a in te['annotations'] if a['category_id'] == unk)
    print(f'{task_name}/test : {len(te["images"]):5d} imgs  '
          f'{len(te["annotations"]):6d} anns  '
          f'(known={len(te["annotations"])-n_unk}, unknown={n_unk})')

    # ── Symlinks ──────────────────────────────────────────────────────────────
    # IMAGE_ROOT = classification/ (thư mục cha)
    # file_name JSON = 'train/...' → full = classification/train/... ✓
    for lname in ['train', 'test']:
        lp = img_dir / lname
        if lp.is_symlink(): lp.unlink()
        elif lp.exists(): shutil.rmtree(lp)
        os.symlink(IMAGE_ROOT, lp)

print('\nChuẩn bị dataset xong.')

t1/train:  5279 imgs    5279 anns  27 classes  (ratio=0.4)
t1/test :  8924 imgs    8924 anns  (known=2652, unknown=6272)

Chuẩn bị dataset xong.


## Bước 4 – Tính MAX_ITER (1 epoch)

In [6]:
def count_imgs(task):
    with open(WORK_DATASETS / task / 'annotations' / 'train.json') as f:
        return len(json.load(f)['images'])

def epoch_iters(task):
    return max(1, math.ceil(count_imgs(task) / IMS_PER_BATCH))

for t in ACTIVE_TASKS:
    n = count_imgs(t)
    it = epoch_iters(t)
    est_min = it * 4 / 60  # ước tính ~4s/iter với 2 GPU
    print(f'{t}: {n} imgs → {it} iters/epoch @ batch={IMS_PER_BATCH}'
          f'  (~{est_min:.0f} phút ước tính)')

t1: 5279 imgs → 294 iters/epoch @ batch=18  (~20 phút ước tính)


## Bước 5 – Hàm train

In [7]:
def resolve_w(w):
    return str(w) if w else 'detectron2://ImageNetPretrained/torchvision/R-50.pkl'

def train_task(task_name, weights):
    """
    Train 1 epoch. Eval hoàn toàn bị tắt trong quá trình train.
    (EvalHook đã bị patch ở Bước 0; EVAL_PERIOD lớn hơn MAX_ITER để chắc chắn)
    """
    info     = TASKS[task_name]
    max_iter = epoch_iters(task_name)
    step1    = max(1, int(max_iter * 0.80))
    step2    = max(step1 + 1, int(max_iter * 0.95))
    out_dir  = OUTPUT_ROOT / task_name
    out_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, 'train_net.py',
        '--num-gpus',    str(NUM_GPUS),
        '--task',        task_name,
        '--config-file', info['config'],
        'MODEL.WEIGHTS',             resolve_w(weights),
        'MODEL.RandBox.NUM_CLASSES', str(NUM_CLASSES_TOTAL),
        'TEST.PREV_INTRODUCED_CLS',  str(info['prev_cls']),
        'TEST.CUR_INTRODUCED_CLS',   str(info['curr_cls']),
        'SOLVER.IMS_PER_BATCH',      str(IMS_PER_BATCH),
        'SOLVER.MAX_ITER',           str(max_iter),
        'SOLVER.STEPS',              f'({step1},{step2})',
        'SOLVER.CHECKPOINT_PERIOD',  str(max_iter),   # lưu 1 lần duy nhất ở cuối
        'TEST.EVAL_PERIOD',          '999999',         # backup: không bao giờ trigger
        'SOLVER.AMP.ENABLED',        'True',
        'DATALOADER.NUM_WORKERS',    str(NUM_WORKERS),
        'OUTPUT_DIR',                str(out_dir),
    ]
    print(f'\n=== TRAIN {task_name} | {max_iter} iters ===')
    print(f'    weights = {resolve_w(weights)[:80]}')
    run(cmd, cwd=REPO_DIR)

    # Tìm checkpoint vừa lưu
    final = out_dir / 'model_final.pth'
    if final.exists(): return final
    cands = sorted(out_dir.glob('model_*.pth'), key=lambda p: p.stat().st_mtime)
    if cands: return cands[-1]
    raise FileNotFoundError(f'Không có checkpoint trong {out_dir}')


print('train_task sẵn sàng.')

train_task sẵn sàng.


## Bước 6 – Training (chỉ train, không eval)

Training sẽ kết thúc khi xong 1 epoch và lưu checkpoint.  
Không có bước eval sau train.

In [8]:
current_w = PRETRAIN_WEIGHTS
trained   = {}

for task_name in ACTIVE_TASKS:
    ckpt = train_task(task_name, current_w)
    trained[task_name] = ckpt
    current_w = ckpt
    print(f'\u2713 {task_name} \u2192 {ckpt}')

print('\n=== Training hoàn tất ===')
print('Checkpoint đã lưu:')
for t, ckpt in trained.items():
    print(f'  {t}: {ckpt}')


=== TRAIN t1 | 294 iters ===
    weights = /kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.p
>>> /usr/bin/python3 train_net.py --num-gpus 2 --task t1 --config-file configs/t1.yaml MODEL.WEIGHTS /kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.pth MODEL.RandBox.NUM_CLASSES 103 TEST.PREV_INTRODUCED_CLS 0 TEST.CUR_INTRODUCED_CLS 27 SOLVER.IMS_PER_BATCH 18 SOLVER.MAX_ITER 294 SOLVER.STEPS (235,279) SOLVER.CHECKPOINT_PERIOD 294 TEST.EVAL_PERIOD 999999 SOLVER.AMP.ENABLED True DATALOADER.NUM_WORKERS 4 OUTPUT_DIR /kaggle/working/randbox_outputs/t1


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Command Line Args: Namespace(config_file='configs/t1.yaml', resume=False, eval_only=False, num_gpus=2, num_machines=1, machine_rank=0, dist_url='tcp://127.0.0.1:49152', opts=['MODEL.WEIGHTS', '/kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.pth', 'MODEL.RandBox.NUM_CLASSES', '103', 'TEST.PREV_INTRODUCED_CLS', '0', 'TEST.CUR_INTRODUCED_CLS', '27', 'SOLVER.IMS_PER_BATCH', '18', 'SOLVER.MAX_ITER', '294', 'SOLVER.STEPS', '(235,279)', 'SOLVER.CHECKPOINT_PERIOD', '294', 'TEST.EVAL_PERIOD', '999999', 'SOLVER.AMP.ENABLED', 'True', 'DATALOADER.NUM_WORKERS', '4', 'OUTPUT_DIR', '/kaggle/working/randbox_outputs/t1'], task='t1')


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


[07/10 10:08:32 detectron2]: Rank of current process: 0. World size: 2
[07/10 10:08:32 detectron2]: Environment info:
-------------------------------  -----------------------------------------------------------------
sys.platform                     linux
Python                           3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
numpy                            2.0.2
detectron2                       0.6 @/kaggle/working/detectron2_src/detectron2
Compiler                         GCC 11.4
CUDA compiler                    CUDA 12.8
detectron2 arch flags            7.5
DETECTRON2_ENV_MODULE            <not set>
PyTorch                          2.10.0+cu128 @/usr/local/lib/python3.12/dist-packages/torch
PyTorch debug build              False
torch._C._GLIBCXX_USE_CXX11_ABI  True
GPU available                    Yes
GPU 0,1                          Tesla T4 (arch=7.5)
Driver version                   580.159.04
CUDA_HOME                        /usr/local/cuda
Pillow                 

...

[07/10 10:12:37 d2.utils.events]:  eta: 0:08:46  iter: 79  total_loss: 16.4  loss_ce: 2.236  loss_bbox: 0.1225  loss_giou: 0.04822  loss_nc_ce: 0.1251  loss_ce_0: 2.46  loss_bbox_0: 0.283  loss_giou_0: 0.1099  loss_nc_ce_0: 0.1109  loss_ce_1: 2.468  loss_bbox_1: 0.2209  loss_giou_1: 0.08654  loss_nc_ce_1: 0.1179  loss_ce_2: 2.448  loss_bbox_2: 0.1641  loss_giou_2: 0.06453  loss_nc_ce_2: 0.1128  loss_ce_3: 2.227  loss_bbox_3: 0.1438  loss_giou_3: 0.05671  loss_nc_ce_3: 0.1305  loss_ce_4: 2.413  loss_bbox_4: 0.1254  loss_giou_4: 0.04919  loss_nc_ce_4: 0.129    time: 2.4739  last_time: 2.6495  data_time: 0.1310  last_data_time: 0.1442   lr: 2.5e-07  max_mem: 14112M


/kaggle/working/detectron2_src/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/kaggle/working/detectron2_src/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/kaggle/working/detectron2_src/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/kaggle/working/detectron2_src/detectron2/engine/train_loop.py:490: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=self.precision):
/kaggle/working/detectron2_src/detectron2/engine/train_loop.py:490: FutureWarnin

ERROR [07/10 10:12:46 d2.engine.train_loop]: Exception during training:
Traceback (most recent call last):
  File "/kaggle/working/detectron2_src/detectron2/engine/train_loop.py", line 152, in train
    self.run_step()
  File "/kaggle/working/detectron2_src/detectron2/engine/defaults.py", line 530, in run_step
    self._trainer.run_step()
  File "/kaggle/working/detectron2_src/detectron2/engine/train_loop.py", line 501, in run_step
    self.grad_scaler.scale(losses).backward()
  File "/usr/local/lib/python3.12/dist-packages/torch/_tensor.py", line 630, in backward
    torch.autograd.backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/__init__.py", line 364, in backward
    _engine_run_backward(
  File "/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py", line 865, in _engine_run_backward
    return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[rank0]:[W710 10:12:47.587469786 ProcessGroupNCCL.cpp:1553] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())
[rank1]:[W710 10:12:48.677080789 TCPStore.cpp:125] [c10d] recvValue failed on SocketImpl(fd=32, addr=[::ffff:127.0.0.1]:49156, remote=[::ffff:127.0.0.1]:49152): Failed to recv, got 0 bytes. Connection was likely closed. Did the remote server shutdown or crash?
Exception raised from recvBytes at /pytorch/torch/csrc/distributed/c10d/Utils.hpp:682 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::__cxx11::basic_string<char, std::char_traits<char>, std::allocator<char> >) + 0x9d (0x7a3196f72fdd in /usr/local/lib/python3.12/dist-packages/torch/lib/libc10.so)
frame #1: <unknown function> + 0x6a3325d (0x7a30f5d8425d in /usr/local/lib/python3.12/dist-packages/torch/lib/libtorch_cpu.so)
frame #2: c10

CalledProcessError: Command '['/usr/bin/python3', 'train_net.py', '--num-gpus', '2', '--task', 't1', '--config-file', 'configs/t1.yaml', 'MODEL.WEIGHTS', '/kaggle/input/models/chienkhu/randbox-pretrain/pytorch/default/1/model_0019999.pth', 'MODEL.RandBox.NUM_CLASSES', '103', 'TEST.PREV_INTRODUCED_CLS', '0', 'TEST.CUR_INTRODUCED_CLS', '27', 'SOLVER.IMS_PER_BATCH', '18', 'SOLVER.MAX_ITER', '294', 'SOLVER.STEPS', '(235,279)', 'SOLVER.CHECKPOINT_PERIOD', '294', 'TEST.EVAL_PERIOD', '999999', 'SOLVER.AMP.ENABLED', 'True', 'DATALOADER.NUM_WORKERS', '4', 'OUTPUT_DIR', '/kaggle/working/randbox_outputs/t1']' returned non-zero exit status 1.

---
## Tùy chọn: Chạy evaluation riêng (nếu cần)

Cell dưới đây **không chạy tự động**.  
Chỉ chạy khi bạn muốn đánh giá checkpoint đã lưu.

In [ ]:
# ?? OPTIONAL: ch? ch?y cell n?y khi c?n eval ??
# Thay ??i path n?u c?n
EVAL_CHECKPOINTS = trained  # ho?c set th? c?ng:
# EVAL_CHECKPOINTS = {'t1': Path('/kaggle/input/models/chienkhu/task-1-train/pytorch/default/1/model_final.pth')}


def patch_coco_json_evaluator(code):
    """Idempotent patch for train_net.py.
    - Ensures DatasetEvaluators + OpenWorldCOCOEvaluator are imported.
    - Does NOT rewrite build_evaluator; the repo version is already correct
      (uses OpenWorldCOCOEvaluator only, no COCOEvaluator that would crash
      with class=33 AssertionError on cross-task predictions).
    """
    # -- Import patch: add DatasetEvaluators + OpenWorldCOCOEvaluator (idempotent) --
    if 'DatasetEvaluators' not in code:
        code = code.replace(
            'from detectron2.evaluation import COCOEvaluator, LVISEvaluator, verify_results',
            'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results'
        )
    if 'open_world_coco_evaluation import OpenWorldCOCOEvaluator' not in code:
        lines = code.splitlines(keepends=True)
        new_lines = []
        for _l in lines:
            new_lines.append(_l)
            if _l.strip() == 'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results':
                new_lines.append('from randbox.open_world_coco_evaluation import OpenWorldCOCOEvaluator\n')
        code = ''.join(new_lines)
    return code
def ensure_eval_batch_patch():
    tn_path = REPO_DIR / 'train_net.py'
    code = tn_path.read_text(encoding='utf-8')
    # -- Import patch: add DatasetEvaluators + OpenWorldCOCOEvaluator (idempotent) --
    if 'DatasetEvaluators' not in code:
        code = code.replace(
            'from detectron2.evaluation import COCOEvaluator, LVISEvaluator, verify_results',
            'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results'
        )
    if 'open_world_coco_evaluation import OpenWorldCOCOEvaluator' not in code:
        lines = code.splitlines(keepends=True)
        new_lines = []
        for _l in lines:
            new_lines.append(_l)
            if _l.strip() == 'from detectron2.evaluation import COCOEvaluator, DatasetEvaluators, LVISEvaluator, verify_results':
                new_lines.append('from randbox.open_world_coco_evaluation import OpenWorldCOCOEvaluator\n')
        code = ''.join(new_lines)

    # Remove older patch/import variants, then add the imports needed by a real batched test loader.
    code = code.replace(
        'from detectron2.data import build_detection_train_loader, build_detection_test_loader',
        'from detectron2.data import build_detection_train_loader'
    )
    code = code.replace(
        'from detectron2.data.common import DatasetFromList, MapDataset, trivial_batch_collator\n',
        'from detectron2.data.common import DatasetFromList, MapDataset\nfrom detectron2.data.build import trivial_batch_collator\n'
    )
    if 'get_detection_dataset_dicts' not in code:
        code = code.replace(
            'from detectron2.data import build_detection_train_loader',
            'from detectron2.data import build_detection_train_loader, get_detection_dataset_dicts'
        )
    if 'from detectron2.data.build import trivial_batch_collator' not in code:
        code = code.replace(
            'from detectron2.data.datasets.coco import load_coco_json\n',
            'from detectron2.data.datasets.coco import load_coco_json\n'
            'from detectron2.data.common import DatasetFromList, MapDataset\n'
            'from detectron2.data.build import trivial_batch_collator\n'
            'from detectron2.data.samplers import InferenceSampler\n'
        )
    if 'cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED]' not in code:
        code = code.replace(
            '    add_model_ema_configs(cfg)\n    cfg.merge_from_file(args.config_file)',
            '    add_model_ema_configs(cfg)\n    cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED] default eval batch size\n    cfg.merge_from_file(args.config_file)'
        )

    old_loader_starts = [
        '    @classmethod\n    def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Use TEST.IMS_PER_BATCH for eval-only.\n',
        '    @classmethod\n    def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Force real batched eval loader.\n',
    ]
    opt_start = '    @classmethod\n    def build_optimizer(cls, cfg, model):'
    for old_start in old_loader_starts:
        if old_start in code:
            start = code.index(old_start)
            end = code.index(opt_start, start)
            code = code[:start] + code[end:]

    new_loader = (
        '    @classmethod\n'
        '    def build_test_loader(cls, cfg, dataset_name):\n'
        '        # [PATCHED] Force real batched eval loader.\n'
        '        dataset_dicts = get_detection_dataset_dicts(dataset_name, filter_empty=False)\n'
        '        mapper = RandBoxDatasetMapper(cfg, is_train=False)\n'
        '        dataset = MapDataset(DatasetFromList(dataset_dicts, copy=False), mapper)\n'
        '        world_size = max(1, comm.get_world_size())\n'
        '        total_batch = max(1, int(cfg.TEST.IMS_PER_BATCH))\n'
        '        per_gpu_batch = max(1, total_batch // world_size)\n'
        '        sampler = InferenceSampler(len(dataset))\n'
        '        batch_sampler = torch.utils.data.sampler.BatchSampler(\n'
        '            sampler, per_gpu_batch, drop_last=False\n'
        '        )\n'
        '        logger = logging.getLogger(__name__)\n'
        '        logger.info(\n'
        '            f"Eval loader: TEST.IMS_PER_BATCH={total_batch}, "\n'
        '            f"world_size={world_size}, per_gpu_batch={per_gpu_batch}, "\n'
        '            f"num_images={len(dataset)}, num_batches_per_rank={len(batch_sampler)}"\n'
        '        )\n'
        '        return torch.utils.data.DataLoader(\n'
        '            dataset,\n'
        '            num_workers=cfg.DATALOADER.NUM_WORKERS,\n'
        '            batch_sampler=batch_sampler,\n'
        '            collate_fn=trivial_batch_collator,\n'
        '        )\n\n'
        '    @classmethod\n'
        '    def build_optimizer(cls, cfg, model):'
    )
    code = code.replace(opt_start, new_loader)

    code = patch_coco_json_evaluator(code)
    tn_path.write_text(code, encoding='utf-8')
    patched = tn_path.read_text(encoding='utf-8')
    required = [
        'cfg.TEST.IMS_PER_BATCH = 1  # [PATCHED]',
        'def build_test_loader(cls, cfg, dataset_name):\n        # [PATCHED] Force real batched eval loader.',
        'BatchSampler(',
    ]
    missing = [s for s in required if s not in patched]
    if missing:
        raise RuntimeError('Patch eval batch ch?a ?p d?ng ??: ' + repr(missing))
    print('Eval batch patch OK: real batched test loader is enabled')

ensure_eval_batch_patch()

def eval_task(task_name, weights):
    info    = TASKS[task_name]
    out_dir = OUTPUT_ROOT / task_name / 'eval'
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, 'train_net.py',
        '--num-gpus',    str(NUM_GPUS),
        '--task',        task_name,
        '--config-file', TASKS[task_name]['config'],
        '--eval-only',
        'MODEL.WEIGHTS',             str(weights),
        'MODEL.RandBox.NUM_CLASSES', str(NUM_CLASSES_TOTAL),
        'TEST.PREV_INTRODUCED_CLS',  str(info['prev_cls']),
        'TEST.CUR_INTRODUCED_CLS',   str(info['curr_cls']),
        'TEST.IMS_PER_BATCH',        str(IMS_PER_BATCH),
        'DATALOADER.NUM_WORKERS',    str(NUM_WORKERS),
        'SOLVER.AMP.ENABLED',        'True',
        'OUTPUT_DIR',                str(out_dir),
    ]
    print(f'=== EVAL {task_name} ===')
    run(cmd, cwd=REPO_DIR)

for task_name, ckpt in EVAL_CHECKPOINTS.items():
    if ckpt and Path(str(ckpt)).exists():
        eval_task(task_name, ckpt)
    else:
        print(f'[SKIP] {task_name}: checkpoint không tồn tại')

# Đọc kết quả
METRICS = [
    ('open_world/known_mAP50',          'Known mAP@50'),
    ('open_world/unknown_mAP50',        'Unknown mAP@50'),
    ('open_world/previous_known_mAP50', 'Previous Known mAP@50'),
    ('open_world/current_known_mAP50',  'Current Known mAP@50'),
    ('open_world/known_recall50',       'Known Recall@50'),
    ('open_world/unknown_recall50',     'Unknown Recall@50'),
    ('open_world/known_precision50',    'Known Precision@50'),
]
print('\n' + '='*65)
for t in EVAL_CHECKPOINTS:
    f = OUTPUT_ROOT / t / 'eval' / 'metrics.json'
    if not f.exists(): continue
    lines = [l.strip() for l in open(f) if l.strip()]
    m = json.loads(lines[-1]) if lines else {}
    print(f'\n-- {t} --')
    for key, label in METRICS:
        v = m.get(key)
        print(f'  {label:<30s}: {v:.2f}' if v is not None else f'  {label:<30s}: N/A')
print('='*65)

In [ ]:
# ══ EVAL với batch size = train (12) ══
# import subprocess, sys
# from pathlib import Path

# task_name = 't1'   # ← đổi task nếu cần
# ckpt      = '/kaggle/working/randbox_outputs/t1/model_final.pth'  # ← đổi path nếu cần

# cmd = [
#     sys.executable, 'train_net.py',
#     '--num-gpus',    '2',
#     '--task',        task_name,
#     '--config-file', f'configs/{task_name}.yaml',
#     '--eval-only',
#     'MODEL.WEIGHTS',             ckpt,
#     'MODEL.RandBox.NUM_CLASSES', '103',
#     'TEST.PREV_INTRODUCED_CLS',  '0',
#     'TEST.CUR_INTRODUCED_CLS',   '27',
#     'TEST.IMS_PER_BATCH',        '12',   # ← bằng train batch
#     'DATALOADER.NUM_WORKERS',    '8',
#     'SOLVER.AMP.ENABLED',        'True',
#     'OUTPUT_DIR',                f'/kaggle/working/randbox_outputs/{task_name}/eval',
# ]

# print('Lệnh:', ' '.join(cmd))
# subprocess.run(cmd, cwd='/kaggle/working/RandBox', check=True)
